# 프롬프트 캐시 계측

동일한 긴 접두어를 두 번 전송했을 때 실제 사용량에서 캐시 읽기 토큰을 구분할 수 있는지 확인합니다.

## 실행 환경 준비

OpenAI SDK와 `.env`를 준비합니다. 키 값은 출력하지 않습니다.

### 이 준비 셀이 하는 일

준비 셀은 현재 폴더와 부모 폴더에서 `ch34-cost-rate-limits` 실습 폴더를 찾고, 발견한 `.env`가 있으면 현재 프로세스에 불러옵니다. 키 값은 출력하지 않으며 준비 결과로 폴더와 환경 파일의 존재 여부만 확인합니다. 이후 파일 기록은 찾은 `chapter_root` 아래 `runs` 폴더를 사용합니다.

In [1]:
from pathlib import Path
from dotenv import load_dotenv

here = Path.cwd().resolve()
chapter_root = next((p for p in (here, *here.parents) if p.name == "ch34-cost-rate-limits" and (p / "notebooks").is_dir()), None)
if chapter_root is None:
    raise FileNotFoundError("ch34-cost-rate-limits 실습 폴더 안에서 실행하세요.")
env_path = next((p / ".env" for p in (here, *here.parents) if (p / ".env").exists()), None)
if env_path:
    load_dotenv(env_path)
print("Python 준비 완료", "API 키 파일 확인" if env_path else "API 키 파일 없음")

Python 준비 완료 API 키 파일 확인


## 긴 공통 접두어 만들기

프롬프트 캐시는 1,024토큰 이상의 접두어에서 동작하므로 반복해서 사용할 안정적인 앞부분을 만듭니다.

In [2]:
import uuid

policy = "한국경제신문 고객지원 정책: 사실과 추정을 구분하고, 근거가 없으면 확인이 필요하다고 답합니다. "
stable_prefix = f"실험 ID: {uuid.uuid4()}\n" + policy * 420
questions = ["환불 문의를 한 문장으로 분류하세요.", "배송 문의를 한 문장으로 분류하세요."]
print("공통 접두어 문자 수:", len(stable_prefix))

공통 접두어 문자 수: 22724


## 실제 API를 두 번 호출하기

모델과 공통 접두어는 유지하고 마지막 질문만 바꿉니다. 각 응답의 `input_tokens_details.cached_tokens`를 읽습니다.

### 바로 다음 코드를 읽는 방법

프롬프트 캐시는 입력의 처음부터 같은 토큰 구간을 다시 사용할 때 적용됩니다. 예제는 고정 정책을 긴 `stable_prefix`로 만들고 마지막 질문만 바꿔 두 번 호출합니다. `usage_metadata`의 전체 입력 토큰과 `input_token_details`의 캐시 읽기 값을 따로 기록하므로, 두 번째 호출이 빨랐다는 느낌 대신 제공자가 반환한 사용량으로 적중을 판단합니다.

In [3]:
import json
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none", max_tokens=40, max_retries=1, seed=34)
records = []
for question in questions:
    answer = model.invoke(stable_prefix + "\n" + question)
    usage = answer.usage_metadata or {}
    details = usage.get("input_token_details") or {}
    cached = details.get("cache_read") or 0
    creation = details.get("cache_creation")
    records.append({"text": answer.text, "usage": usage, "input": usage.get("input_tokens"),
                    "cached_read": cached, "cache_creation": creation, "output": usage.get("output_tokens")})
run_dir = chapter_root / "runs"; run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "cache-measurement.json").write_text(json.dumps(records, ensure_ascii=False, indent=2))
records

[{'text': '분류: **환불 문의**\n\n근거가 충분하지 않아 보이므로, 세부 유형(예: 구독 해지 후 환불, 중복 결제 환',
  'usage': {'input_tokens': 11805,
   'output_tokens': 40,
   'total_tokens': 11845,
   'input_token_details': {'audio': 0, 'cache_read': 0},
   'output_token_details': {'audio': 0, 'reasoning': 0}},
  'input': 11805,
  'cached_read': 0,
  'cache_creation': None,
  'output': 40},
 {'text': '배송 문의로 **분류됩니다**.  \n\n- **사실:** 사용자가 “배송 문의를 한 문장으로 분류”해 달라고 요청했습니다.\n- **추',
  'usage': {'input_tokens': 11804,
   'output_tokens': 40,
   'total_tokens': 11844,
   'input_token_details': {'audio': 0, 'cache_read': 11520},
   'output_token_details': {'audio': 0, 'reasoning': 0}},
  'input': 11804,
  'cached_read': 11520,
  'cache_creation': None,
  'output': 40}]

## read와 creation을 분리해 읽기

`cached_read`는 실제 사용량 메타데이터에서 얻습니다. 현재 사용한 LangChain `usage_metadata`에서 별도 생성 값을 얻지 못하면 `None`으로 기록합니다.

### 이 단계에서 구분할 범위

문장 내용이 비슷해도 앞부분의 토큰 배열이 달라지면 같은 접두어가 아닙니다. 날짜나 사용자 식별자를 맨 앞에 넣으면 뒤의 긴 정책까지 재사용되지 않을 수 있습니다. 반대로 캐시 적중은 답의 품질이나 최신성을 높이지 않으므로, 반복 지시문 안정성과 질문별 근거 갱신을 별도로 관리합니다.

In [4]:
for index, row in enumerate(records, 1):
    print(f"호출 {index}: input={row['input']}, cache_read={row['cached_read']}, cache_creation={row['cache_creation']}")
assert records[1]["cached_read"] >= 1024
assert all(row["text"] and row["output"] > 0 for row in records)
print("두 번째 호출에서 캐시 읽기 확인")

호출 1: input=11805, cache_read=0, cache_creation=None
호출 2: input=11804, cache_read=11520, cache_creation=None
두 번째 호출에서 캐시 읽기 확인


## 실험 결과 해석

캐시 적중 여부는 시간 추정이나 파일 수가 아니라 API가 반환한 사용량으로 판정합니다. 생성 토큰이 제공되지 않았다는 사실과 0을 구분합니다.

In [5]:
uncached = [row["input"] - row["cached_read"] for row in records]
print("비캐시 입력 토큰:", uncached)
print("캐시 생성 계측 지원:", records[0]["cache_creation"] is not None)

비캐시 입력 토큰: [11805, 284]
캐시 생성 계측 지원: False
